In [1]:
%pip install -q google-adk


Note: you may need to restart the kernel to use updated packages.


In [2]:
from google.adk.agents import LlmAgent, SequentialAgent

print("Google ADK is ready!")

Google ADK is ready!


In [3]:
import os
from getpass import getpass
from typing import Literal
from pydantic import BaseModel, Field
from google.genai import Client, types

# Prompt you to enter your Gemini API key
api_key = getpass("Enter your Gemini API key: ")

# Configure Google ADK
os.environ["GOOGLE_API_KEY"] = api_key
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "FALSE"

print("Gemini API key configured successfully!")

Gemini API key configured successfully!


In [4]:
import pandas as pd

full_df = pd.read_csv("../data/baseline_50_full.csv")
gemini_df = pd.read_csv("../data/baseline_50_gemini_outputs.csv")

print("Both files loaded successfully!")

Both files loaded successfully!


In [5]:
# TODO (ADK pipeline): read inputs from shared state with {placeholders} in the instruction,
#   e.g. {source}, {facts} (Extractor), {simplified} and {retrieved_passages} (Simplifier).
#   Key names must match the other agents' output_key values; agree on them with the team.
# TODO (ADK pipeline): set output_schema=VerifierJudgment so the agent returns JSON in state["verification"].
# TODO (Simplifier): add a "background" label to the instruction: general info not in the source
#   counts as supported if a RETRIEVED BACKGROUND (MedlinePlus) passage backs it.
# TODO (Extractor): tell the agent to check each KEY FACT and report missing ones as omissions by fact_id.
verifier_agent = LlmAgent(
    name="Verifier",
    model="gemini-3.5-flash-lite",
    description="Checks patient-friendly medical explanations for accuracy and completeness.",
    instruction="""
    You are a medical verification agent.

    Your task is to compare a patient-friendly medical
    explanation against the original medical source.

    Check for:
    1. Incorrect or unsupported medical claims.
    2. Contradictions with the original source.
    3. Missing important medical information.
    4. Incorrect medication names, doses, or instructions.
    5. Incorrect numbers, test results, or follow-up dates.
    6. Changes in medical risk, severity, or uncertainty.

    Break the patient-friendly explanation into individual
    factual medical claims.

    Check every claim separately against the original source.

    Include all claims in your evaluation, even those that
    are correct. Label each claim as supported, unsupported,
    or distorted.

    A claim that changes a fact from the source (a number,
    unit, timing, or a hedge turned into certainty) is
    distorted.

    Pay special attention to negation errors.

    Compare statements containing words such as
    "no", "not", "never", "without", and "negative"
    against the original source.

    If the source reports a symptom or finding,
    but the explanation says it is absent,
    mark the claim as distorted.

    Check each part of a sentence separately when
    it contains multiple medical facts.

    For every unsupported or contradicted claim,
    provide the relevant evidence from the original source.

    If the source does not mention the claim, clearly state:
    "Not mentioned in the original source."

    For each issue:
    - Identify the incorrect or missing information.
    - Explain why it is a problem.
    - Provide evidence from the original source.
    - Indicate whether the issue is minor or major.

    Important rules:
    - Use only the original medical source to verify facts.
    - Do not invent medical information.
    - Do not treat suspected conditions as confirmed diagnoses.
    - Prioritize errors that could affect patient safety.
    - Check the entire explanation, not just a few claims.
    - Do not overlook a contradiction just because most
      of the explanation is accurate.

    Provide clear feedback so the Refiner Agent
    can correct the explanation.
    """,
    output_key="verification"
)

print("Verifier Agent created!")

Verifier Agent created!


In [6]:
Severity = Literal["minor", "major"]

class ClaimCheck(BaseModel):
    claim: str = Field(description="One specific statement from the patient explanation")
    # TODO (Simplifier): add "background" once MedlinePlus passage text is available
    label: Literal["supported", "unsupported", "distorted"] = Field(
        description="supported: the source states it; unsupported: the source does not state it; "
                    "distorted: changes a fact from the source (a number, unit, timing, or a hedge turned into certainty)")
    evidence: str = Field(description="Exact quote from the source that supports or contradicts the claim, or '' if none")
    severity: Severity = Field(description="The clinical importance of claim being incorrect.")

class Omission(BaseModel):
    # TODO (Extractor): add fact_id: str so each omission points to an Extractor fact
    fact: str = Field(description="An important fact from the source that the explanation leaves out")
    severity: Severity = Field(description="The clinical importance of omission being made.")

class VerifierJudgment(BaseModel):
    claims: list[ClaimCheck]
    omissions: list[Omission]

class VerifierReport(VerifierJudgment):
    faithfulness_score: float = Field(description="Fraction of claims that are supported")
    omission_count: int = Field(description="Important source facts missing")
    addition_count: int = Field(description="Claims labelled unsupported (not in the source)")
    hallucination: bool = Field(description="At least one major unsupported or distorted claim.")
    needs_revision: bool = Field(description="Hallucination, or any major omission. Sent to refiner agent.")
    # TODO (Simplifier): add false_coverage: list[str], fact_ids in facts_covered that are actually missing

def build_report(judgment: VerifierJudgment) -> VerifierReport:
    # TODO (Simplifier): count "background" as supported AND as an addition (advisor's Rewrite A)
    supported = [c for c in judgment.claims if c.label == "supported"]
    additions = [c for c in judgment.claims if c.label == "unsupported"]
    hallucinated = [c for c in judgment.claims if c.label in ("unsupported", "distorted") and c.severity == "major"]
    serious_omissions = [o for o in judgment.omissions if o.severity == "major"]

    n = len(judgment.claims)
    hallucination = len(hallucinated) > 0
    # TODO (Simplifier): compute false_coverage = omitted fact_ids & set(facts_covered)
    # TODO (ADK pipeline): decide where build_report runs (e.g. the LoopAgent stop check before the Refiner)

    return VerifierReport(
        **judgment.model_dump(),
        faithfulness_score=len(supported) / n if n else 1.0,
        omission_count=len(judgment.omissions),
        addition_count=len(additions),
        hallucination=hallucination,
        needs_revision = hallucination or len(serious_omissions) > 0,
    )

### Initial Test

In [7]:
client = Client()

def create_verifier_chat():
    config = types.GenerateContentConfig(
        system_instruction=verifier_agent.instruction,
        response_mime_type="application/json",
        response_schema=VerifierJudgment,
        temperature=0,
    )
    return client.chats.create(model=verifier_agent.model, config=config)

# TODO (Extractor/Simplifier): add inputs facts (Extractor), passages and facts_covered (Simplifier)
def run_verifier_agent(source: str, explanation: str) -> VerifierReport:
    # new chat for every check, so the judge never sees earlier drafts
    chat = create_verifier_chat()

    # TODO (Extractor/Simplifier): add KEY FACTS and RETRIEVED BACKGROUND sections to the message
    # generate response
    response = chat.send_message(f"SOURCE:\n{source}\n\nEXPLANATION:\n{explanation}")

    # parse pydantic model as JSON, then compute the numbers in code
    judgment = VerifierJudgment.model_validate_json(response.text)
    report = build_report(judgment)

    # print output
    print("Verifier Report")
    print(f"Faithfulness score: {report.faithfulness_score:.2f}")
    print(f"Omissions: {report.omission_count} | Additions: {report.addition_count}")
    print(f"Hallucination: {report.hallucination} | Needs revision: {report.needs_revision}")
    for c in report.claims:
        if c.label != "supported":
            print(f"  [{c.label}, {c.severity}] {c.claim}")
    for o in report.omissions:
        print(f"  [omitted, {o.severity}] {o.fact}")

    return report

In [8]:
source = """DISCHARGE SUMMARY
Patient: 68-year-old woman. Admitted 10/02, discharged 10/06.
Reason for admission: Shortness of breath and leg swelling due to acute decompensated heart failure with reduced ejection fraction (EF 30%). New-onset atrial fibrillation was also found during the admission.
Hospital course: Treated with IV furosemide with good diuresis; weight decreased by 4.2 kg. Breathing returned to baseline. Atrial fibrillation was rate-controlled with metoprolol. Given a CHA2DS2-VASc score of 4, apixaban was started for stroke prevention. Echocardiogram showed no intracardiac thrombus. Potassium fell to 3.2 during diuresis and was replaced; 4.0 at discharge. Kidney function remained stable (creatinine 1.1).
Nursing noted loud snoring with pauses in breathing overnight, concerning for possible obstructive sleep apnea; this was not evaluated during the stay.
Discharge medications:
- Furosemide 40 mg by mouth once daily (increased from 20 mg)
- Metoprolol succinate 50 mg by mouth once daily (new)
- Apixaban 5 mg by mouth twice daily (new)
- Lisinopril 10 mg by mouth once daily (unchanged)
- Potassium chloride 20 mEq by mouth once daily for 7 days, then stop
- Stop ibuprofen. May use acetaminophen up to 3,000 mg per day for knee pain.
Instructions:
- Weigh yourself every morning after urinating and before breakfast. Call the heart failure clinic if weight increases by more than 2 pounds in one day or 5 pounds in one week.
- Limit sodium to 2,000 mg per day and fluids to 2 liters per day.
- Do not skip apixaban doses. Do not stop apixaban without talking to your cardiologist, as stopping increases the risk of stroke.
- Report any unusual bleeding, black stools, or blood in the urine.
- Go to the emergency department for chest pain, fainting, or shortness of breath at rest.
Follow-up:
- Basic metabolic panel (potassium and kidney function) in 1 week at the outpatient lab.
- Heart failure clinic in 7 to 10 days.
- Cardiology in 4 weeks to discuss long-term heart rhythm management.
- Primary care to arrange a sleep study."""

draft = """Why you were in the hospital
You came to the hospital because you were short of breath and your legs were swollen. This happened because your heart is weak and was not pumping well. This is called heart failure. Your heart is pumping at about 30% of its normal strength. Doctors also found that you have an irregular heartbeat called atrial fibrillation.

What happened in the hospital
You got a water pill through an IV to remove extra fluid, and you lost about 4 pounds. Your breathing went back to normal. You started a medicine to slow your heart rate and a blood thinner to lower your chance of a stroke. A heart ultrasound showed no blood clots. Your potassium got low and was treated. Your kidneys are working well.

Your medicines
- Furosemide (water pill): 40 mg once a day. This is a higher dose than before.
- Metoprolol: 50 mg once a day to slow your heart rate.
- Apixaban (blood thinner): 5 mg once a day.
- Lisinopril: 10 mg once a day, the same as before.
- Potassium: 20 mEq once a day.
- For knee pain, take acetaminophen (Tylenol) instead of ibuprofen.

What to do at home
- Weigh yourself every morning. Call the clinic if you gain more than 2 pounds in a day or 5 pounds in a week.
- Eat less salt and drink less fluid.
- Take your blood thinner every day. If it gives you side effects, you can stop taking it.
- Call if you notice bleeding or black stools.
- Go to the emergency room if you have chest pain or faint.
- Try to walk 30 minutes a day to make your heart stronger.

Follow-up
- Get a blood test in 1 week.
- See the heart failure clinic in 1 to 2 weeks.
- See your heart doctor in 4 weeks.
- You have sleep apnea, so your primary care doctor will set up a sleep study."""

report = run_verifier_agent(source, draft)

Verifier Report
Faithfulness score: 0.85
Omissions: 3 | Additions: 1
Hallucination: True | Needs revision: True
  [distorted, major] Apixaban (blood thinner): 5 mg once a day.
  [distorted, major] Take your blood thinner every day. If it gives you side effects, you can stop taking it.
  [unsupported, major] Try to walk 30 minutes a day to make your heart stronger.
  [distorted, major] You have sleep apnea, so your primary care doctor will set up a sleep study.
  [omitted, minor] Failing to mention the specific fluid and sodium limits in numbers (2,000 mg sodium and 2 liters fluid per day) in the at-home section.
  [omitted, minor] Omitting the instruction to weigh yourself after urinating and before breakfast.
  [omitted, major] Omitting the instruction to stop potassium after 7 days.


In [9]:
print("Total claims:", len(report.claims))

print("Supported:", sum(
    c.label == "supported" for c in report.claims
))

print("Unsupported:", sum(
    c.label == "unsupported" for c in report.claims
))

print("Distorted:", sum(
    c.label == "distorted" for c in report.claims
))

print("Omissions:", len(report.omissions))

Total claims: 26
Supported: 22
Unsupported: 1
Distorted: 3
Omissions: 3


### Running on dev split of baseline set (Gemini Output)

In [10]:
source = gemini_df.iloc[0]["source_text"]
draft = gemini_df.iloc[0]["gemini_output"]

report = run_verifier_agent(source, draft)

Verifier Report
Faithfulness score: 0.98
Omissions: 1 | Additions: 0
Hallucination: True | Needs revision: True
  [distorted, major] He has been a little fussy, but he does not have a fever, diarrhea, or a history of easy bruising or excessive bleeding from minor cuts.
  [omitted, minor] The patient's Meckel's diverticulum explanation characterizes it as harmless, which is not mentioned in the original source.


In [11]:
# get dev split
dev_split_ids = full_df[full_df.split == 'dev']['row_id'].tolist()
dev_split_df = gemini_df[gemini_df['example_id'].isin(dev_split_ids)]

# loop through dev split
for i, row in dev_split_df.iterrows():
    source = row["source_text"]
    draft = row["gemini_output"]

    print(f'Sample {i+1} | mtsamples id: {row["example_id"]}')
    report = run_verifier_agent(source, draft)
    print("Total claims:", len(report.claims))
    print("Supported:", sum(c.label == "supported" for c in report.claims))
    print("Unsupported:", sum(c.label == "unsupported" for c in report.claims))
    print("Distorted:", sum(c.label == "distorted" for c in report.claims))
    print("Omissions:", len(report.omissions))

    print('\n')

Sample 1 | mtsamples id: 1896
Verifier Report
Faithfulness score: 0.98
Omissions: 0 | Additions: 1
Hallucination: False | Needs revision: False
  [unsupported, minor] Meckel's diverticulum (a small, harmless pouch in the intestine present from birth that can bleed)
Total claims: 46
Supported: 45
Unsupported: 1
Distorted: 0
Omissions: 0


Sample 2 | mtsamples id: 3527
Verifier Report
Faithfulness score: 0.96
Omissions: 1 | Additions: 0
Hallucination: True | Needs revision: True
  [distorted, major] Fever, fatigue, blurred vision, rash, and itching.
  [omitted, minor] The source mentions the patient's ethnicity as Irish-Greek.
Total claims: 24
Supported: 23
Unsupported: 0
Distorted: 1
Omissions: 1


Sample 3 | mtsamples id: 1910
Verifier Report
Faithfulness score: 0.97
Omissions: 0 | Additions: 1
Hallucination: False | Needs revision: False
  [unsupported, minor] Tylenol and ibuprofen can be used as needed for fevers
Total claims: 35
Supported: 34
Unsupported: 1
Distorted: 0
Omissions: 0

In [12]:
print("Full dataset columns:")
print(full_df.columns.tolist())

print("\nGemini outputs columns:")
print(gemini_df.columns.tolist())

Full dataset columns:
['row_id', 'text_hash', 'split', 'doc_type', 'domain', 'specialties', 'n_words', 'length_bin', 'has_meds', 'has_labs', 'sample_name', 'text']

Gemini outputs columns:
['example_id', 'source_text', 'gemini_output', 'model', 'source_grade', 'output_grade', 'grade_improvement', 'smog_index', 'jargon_density', 'word_count', 'meets_grade_8']


### Calibrating against MedAESQA